# 01 — Extração e Limpeza dos Dados de Campo

Meu ponto de partida é o formulário de campo da fazenda: um Excel com uma aba por
viveiro (20 viveiros no total). O problema é que essa planilha foi feita pra ser lida por
gente, não por computador, sendo assim, os cultivos ficam empilhados um embaixo do outro, tem célula
mesclada, número no formato brasileiro e campo em branco preenchido com zero. Então antes
de qualquer análise eu precisei transformar isso numa tabela organizada.

Nesse trabalho meu objetivo é **prever o peso do camarão** ao longo do cultivo. A
biometria (a pesagem) está registrada nesse mesmo formulário, então a limpeza é o
primeiro passo pra chegar até ela.

## Extração

Eu escrevi um módulo (`src/extracao.py`) que varre cada aba procurando os marcos de texto
("CULTIVO Nº", "DIAS", "PESO", "GP") e reconstrói uma tabela com uma linha por semana de
cada cultivo. Um cuidado que tomei foi reiniciar o peso a cada cultivo novo, pra o peso
final de um não vazar pro começo do seguinte, e marcar com a flag `EH_BIOMETRIA` só a
linha onde a pesagem realmente aconteceu.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi']=100; plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=0.3

In [2]:
import subprocess
print(subprocess.run(["python","-m","src.extracao"], capture_output=True, text=True, cwd="..").stdout[-500:])

VEIRO_N_12B1.xlsx
  VIVEIRO Nº 12B2      -> 10 cultivos -> VIVEIRO_N_12B2.xlsx
  VIVEIRO Nº 12B3      ->  9 cultivos -> VIVEIRO_N_12B3.xlsx
  VIVEIRO Nº 12B4      -> 10 cultivos -> VIVEIRO_N_12B4.xlsx
  VIVEIRO Nº 12B5      ->  8 cultivos -> VIVEIRO_N_12B5.xlsx
  VIVEIRO Nº 13        ->  9 cultivos -> VIVEIRO_N_13.xlsx
  VIVEIRO Nº 14        ->  8 cultivos -> VIVEIRO_N_14.xlsx

--------------------------------------------------
Extracao concluida: 20/20 viveiros
Total de cultivos extraidos: 185



## Limpeza

Depois juntei todos os viveiros e apliquei correções com critério biológico. 
A principal foi tratar **zero como valor ausente** nas medições porque na planilha célula vazia
virou 0, mas oxigênio 0 mg/L é impossível, é dado faltante e não medição. 
Também criei a coluna `ORDEM`, que guarda a cronologia de cada cultivo (o campo DIAS tem buracos e não
serve pra ordenar), e algumas colunas derivadas como a densidade de estocagem.

In [3]:
print(subprocess.run(["python","-m","src.limpeza"], capture_output=True, text=True, cwd="..").stdout[-600:])

 unicos:      185
  Biometrias:         2,199

  % de NaN por coluna de medicao:
    SUP_05           21.1%  ####
    SOLO_05          21.1%  ####
    TEMP_05          21.0%  ####
    PH_05            25.0%  ####
    SUP_14           30.0%  #####
    SOLO_14          30.0%  #####
    TEMP_14          29.2%  #####
    PH_14            36.4%  #######
    SAL_14           33.7%  ######
    PESO             31.0%  ######
    GANHO_PESO       33.4%  ######
    UM               18.8%  ###
    DOIS             23.4%  ####
    TDIA             18.8%  ###

Pronto! Dataset disponivel em data/processed/



In [4]:
df = pd.read_csv("../data/processed/dataset_completo.csv", low_memory=False)
print("linhas:", len(df), "| cultivos:", df["ID_CULTIVO"].nunique(), "| viveiros:", df["VIVEIRO"].nunique())
df[["ID_CULTIVO","VIVEIRO","ORDEM","DIAS","PESO","TDIA","EH_BIOMETRIA"]].head(8)

linhas: 22312 | cultivos: 185 | viveiros: 20


,ID_CULTIVO,VIVEIRO,ORDEM,DIAS,PESO,TDIA,EH_BIOMETRIA
0,V1_C1,1,0,NaN,NaN,NaN,0
1,V1_C1,1,1,NaN,NaN,NaN,0
2,V1_C1,1,2,NaN,NaN,NaN,0
3,V1_C1,1,3,NaN,NaN,NaN,0
4,V1_C1,1,4,NaN,NaN,NaN,0
5,V1_C1,1,5,NaN,NaN,NaN,0
6,V1_C1,1,6,NaN,NaN,NaN,0
7,V1_C1,1,7,NaN,NaN,NaN,0


No próximo notebook eu exploro esses dados pra entender o comportamento do peso antes
de modelar.